In [3]:
from pathlib import Path
import pandas as pd, numpy as np, json, matplotlib.pyplot as plt, seaborn as sns
try:
    from wordcloud import WordCloud
    HAS_WC = True
    print("wordcloud available")
except ImportError:
    HAS_WC = False
    print("wordcloud not installed — Fig3 will be bar chart fallback. pip install wordcloud")

BASE = Path("C:/Users/phoen/Code/Repos/jupyter/sm-bias")
RES = BASE / "results"
FIG = BASE / "figures"
FIG.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")


wordcloud available


In [ ]:
summary = pd.read_csv(RES / "flagging_rate_summary.csv")
s07 = summary[summary['threshold']==0.7]
plt.figure(figsize=(8,5))
colors = {"baseline":"#4e79a7","crisis":"#e15759","neutral_generic":"#59a14f","neutral_injected":"#f28e2b"}
plt.bar(s07['split'], s07['FR'], color=[colors.get(x,"grey") for x in s07['split']])
plt.ylabel("Flagging Rate (threshold 0.7)")
plt.title("Flagging Rate: Baseline vs Crisis vs Neutral Probes")
plt.xticks(rotation=15)
for i, v in enumerate(s07['FR']):
    plt.text(i, v+0.01, f"{v:.1%}\n(n={int(s07.iloc[i]['n'])})", ha="center", fontsize=9)
plt.tight_layout()
plt.savefig(FIG / "fig1_flagging_rate.png", dpi=300)
plt.close()
print("Fig1 -> figures/fig1_flagging_rate.png")


In [ ]:
neutral = pd.read_csv(RES / "neutral_scored.csv")
plt.figure(figsize=(8,5))
sns.histplot(data=neutral, x="toxicity", hue="type", bins=15, kde=True, palette=["#59a14f","#f28e2b"], alpha=0.6)
plt.axvline(0.7, color="red", linestyle="--", label="flag threshold 0.7")
plt.title("Toxicity Distribution: Generic vs Crisis-Injected Neutral Statements")
plt.xlabel("Toxicity Score")
plt.legend()
plt.tight_layout()
plt.savefig(FIG / "fig2_neutral_distribution.png", dpi=300)
plt.close()
print("Fig2 -> figures/fig2_neutral_distribution.png")


In [ ]:
if (RES/"crisis_scored.csv").exists():
    crisis = pd.read_csv(RES/"crisis_scored.csv")
    baseline = pd.read_csv(RES/"baseline_scored.csv")
    plt.figure(figsize=(8,5))
    df2 = pd.concat([crisis.assign(split="crisis")[["toxicity","split"]], baseline.assign(split="baseline")[["toxicity","split"]]])
    sns.violinplot(data=df2, x="split", y="toxicity", palette=["#e15759","#4e79a7"])
    plt.title("Toxicity Distribution: Crisis vs Baseline Tweets")
    plt.tight_layout()
    plt.savefig(FIG / "fig2b_crisis_baseline_violin.png", dpi=300)
    plt.close()


In [ ]:
hotspots = pd.read_csv(RES / "top20_hotspots.csv")
if HAS_WC:
    text = " ".join([ (w+" ")*int(max(1, d*1000)) for w, d in zip(hotspots['word'], hotspots['delta_tfidf']) ])
    wc = WordCloud(width=800, height=400, background_color="white", colormap="Reds").generate(text)
    plt.figure(figsize=(10,5))
    plt.imshow(wc, interpolation="bilinear")
    plt.axis("off")
    plt.title("High-Risk Crisis Keywords (TF-IDF delta weighted)")
    plt.tight_layout()
    plt.savefig(FIG / "fig3_hotspot_wordcloud.png", dpi=300)
    plt.close()
else:
    plt.figure(figsize=(10,5))
    plt.barh(hotspots['word'].head(15)[::-1], hotspots['delta_tfidf'].head(15)[::-1], color="#e15759")
    plt.title("High-Risk Crisis Keywords (TF-IDF delta)")
    plt.tight_layout()
    plt.savefig(FIG / "fig3_hotspot_wordcloud.png", dpi=300)
    plt.close()
print("Fig3 -> figures/fig3_hotspot_wordcloud.png")


In [ ]:
if (RES / "decision_boundary_weights.csv").exists():
    w = pd.read_csv(RES / "decision_boundary_weights.csv")
    plt.figure(figsize=(10,6))
    w_sorted = w.sort_values("weight")
    colors2 = ["#4e79a7" if x=="pro-true" else "#e15759" for x in w_sorted['direction']]
    plt.barh(w_sorted['feature'], w_sorted['weight'], color=colors2)
    plt.title("Decision Boundary: Top Pro-Misinfo vs Pro-True Features (Baseline-trained LR)")
    plt.xlabel("LR weight")
    plt.tight_layout()
    plt.savefig(FIG / "fig4_decision_boundary.png", dpi=300)
    plt.close()
    print("Fig4 -> figures/fig4_decision_boundary.png")


In [ ]:
s07_dict = s07.set_index('split')['FR'].to_dict()
snippet = f"""
## Quick Numbers for Paper (auto-generated)

- Baseline FR@0.7: {s07_dict.get('baseline',0):.1%}
- Crisis FR@0.7: {s07_dict.get('crisis',0):.1%} (delta {(s07_dict.get('crisis',0)-s07_dict.get('baseline',0)):+.1%})
- Neutral generic FR@0.7: {s07_dict.get('neutral_generic',0):.1%}
- Neutral injected FR@0.7: {s07_dict.get('neutral_injected',0):.1%} (delta {(s07_dict.get('neutral_injected',0)-s07_dict.get('neutral_generic',0)):+.1%})
- Hotspots: {', '.join(hotspots['word'].head(10))}
"""
(Path(BASE/"to-do/logs/day06_numbers.md")).write_text(snippet, encoding="utf-8")
print(snippet)
print("All figures done. Log -> to-do/logs/day06_numbers.md")
